In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

spark = SparkSession.builder.appName("FinalProject").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 13:19:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 13:19:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/29 13:19:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/29 13:19:11 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/29 13:19:11 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/29 13:19:11 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


# Part I. Fitting Model

## 1. Read training data

In [3]:
#Instruction: You should read this data into a standard pandas data frame using the pd.read_csv() function.
df_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
df_pd.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [4]:
#Instruction: Convert this to a spark data frame
spark_df = spark.createDataFrame(df_pd)
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [5]:
spark_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## 2. Pre-processing

Instruction:
We are going to treat the Power_Zone_3 variable as our response variable.
We can use all of the other variables as predictors. (Imagine we know that the Power_Zone_3 reading
is going to go offline in the future and we need to be able to predict that value appropriately.)

In [6]:
#cast hour and rename response variable
sql_transformer = SQLTransformer(statement = """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double, Power_Zone_3 AS label
    FROM __THIS__""")
#예측변수인 Power_Zone_3를 label이라는 이름으로 복사

Instruction: The Hour column is likely not stored as a DoubleType. If it is not, use an SQL transformer to cast the
variable as a DoubleType.
Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)

In [7]:
#create new binary variable
hour_binarizer = Binarizer(threshold = 6.5, inputCol = "Hour_double", outputCol = "Hour_binary")

Instruction: One-hot encode the Month column

In [8]:
month_encoder = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_encoded"])

Instruction: Run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and
Diffuse_Flows columns.

In [9]:
#Instruction: Use a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator.
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol = "pca_input")

In [10]:
#Instruction: We’ll use two PCs in our transformation.
pca = PCA(k = 2, inputCol = "pca_input", outputCol = "pca_features") #define PCA transformer

Instruction: Use VectorAssembler() to put your predictors into a features. Use the
* two fitted PCA features
* binary Hour variable
* Power_Zone_1
* Power_Zone_2
* Month indicator variables

In [11]:
feature_assembler = VectorAssembler(
    inputCols = ["pca_features", "Hour_binary", "Power_Zone_1", "Power_Zone_2", "Month_encoded"],
    outputCol = "features")

Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.

In [12]:
lr = LinearRegression(featuresCol = "features", labelCol = "label", predictionCol = "prediction")

Instruction: The transformations below should each use an MLlib function that can be put into a pipeline

In [13]:
#Build full pipeline
pipeline = Pipeline(stages = [sql_transformer, hour_binarizer, month_encoder, pca_assembler, pca, feature_assembler, lr])

## 3. Hyperparameter Tuning

Instruction: Now you’ll then use the CrossValidator() function and the LinearRegression() function to fit an
elastic net model.

In [14]:
#define evaluator of model performance
evaluator = RegressionEvaluator(labelCol = "label", predictionCol = "prediction", metricName = "rmse")

Instruction: You should do the following grid for the regParam and elasticNetParam: All combinations of
* regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
* elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

In [15]:
#define parameter grid
param_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]

param_grid = (ParamGridBuilder().addGrid(lr.regParam, param_values).addGrid(lr.elasticNetParam, param_values).build())

Instruction: Now fit the model using 5-fold CV with rmse as your criterion!

In [16]:
#set up 5-fold cross validation
cv = CrossValidator(estimator = pipeline,
                    estimatorParamMaps = param_grid,
                    evaluator = evaluator,
                    numFolds = 5,
                    seed = 123)

In [17]:
#fit the model
cv_model = cv.fit(spark_df)

26/04/29 13:19:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/29 13:19:25 WARN Instrumentation: [3824cb37] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 13:19:26 WARN Instrumentation: [3824cb37] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 13:19:28 WARN Instrumentation: [d7ec32f3] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 13:19:29 WARN Instrumentation: [d7ec32f3] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 13:19:30 WARN Instrumentation: [3587947d] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 13:19:30 WARN Instrumentation: [3587947d] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Instruction: Report the optimal values chosen for the tuning parameters

In [18]:
#report optimal tuning parameters
best_model = cv_model.bestModel
best_lr_model = best_model.stages[-1]

best_reg_param = best_lr_model.getRegParam()
best_elastic_net_param = best_lr_model.getElasticNetParam()

print("Best regParam:", best_reg_param)
print("Best elasticNetParam:", best_elastic_net_param)

Best regParam: 0.05
Best elasticNetParam: 0.1


Instruction: Report the CV error

In [19]:
#cv_model.avgMetrics에는 각 parameter 조합별 평균 CV RMSE가 저장된다. 가장 작은 값이 최적 CV error이다.
cv_errors = cv_model.avgMetrics
best_cv_error = min(cv_errors)

print("Best CV RMSE:", best_cv_error)

Best CV RMSE: 2147.8113505767938


Instruction: Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and
evaluating on the entire training set

In [20]:
#최종 선택된 모델을 training data에 적용
training_predictions = cv_model.transform(spark_df)
training_rmse = evaluator.evaluate(training_predictions)

print("Training RMSE:", training_rmse)

Training RMSE: 2147.0973169293934


Instruction: Take the outputted transformations from the model (the predictions) and create a residual column
(label - prediction). The .withColumn() method is handy here. Print out a data frame with these
residuals, the label column, and the predictions

In [21]:
#예측 결과에서 잔차 칼럼을 만듦
training_predictions_with_residuals = training_predictions.withColumn("residual", F.col("label") - F.col("prediction"))

training_predictions_with_residuals.select("label", "prediction", "residual").show(20)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20878.85066078906|-637.8868007890596|
|20131.08434|18660.227266543865|1470.8570734561363|
|19668.43373|18204.752153114372|1463.6815768856286|
|18899.27711|17590.648498338967|1308.6286116610318|
|18442.40964|16997.301986456696|1445.1076535433058|
|18130.12048| 16517.68672349411|1612.4337565058922|
|17945.06024|16093.246141053307| 1851.814098946692|
|17459.27711|15722.695360253732|1736.5817497462667|
|17025.54217|15271.043828662645|1754.4983413373557|
|16794.21687|14938.348764665461| 1855.868105334539|
|16638.07229|14652.383721423677|1985.6885685763227|
|16395.18072| 14414.90066273023|1980.2800572697706|
|16117.59036|14082.889275686655|2034.7010843133448|
| 15822.6506|13624.882091435917| 2197.768508564084|
|15672.28916|13450.340775468161| 2221.948384531839|
|15597.10843|13302.246394561793|2294.8620354382074|
|15510.36145

# Part II. Streaming Part
There is another file available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv

Download this file and store it where your .py file you’ll create can find it. We’ll be randomly sampling rows from this to output to .csv files that you’ll be reading in.

In [30]:
import pandas as pd
import os

In [31]:
#read sreaming data
stream_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv")

In [32]:
#streaming용 원본 파일 저장
stream_pd.to_csv("power_streaming_data.csv", index = False)
stream_pd.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,4.805,76.2,0.081,0.059,0.134,20421.26582,12908.20669,14590.84337,1,3
1,4.212,78.3,0.081,0.117,0.082,21393.41772,13575.68389,14862.65060,1,5
2,4.304,76.0,0.082,0.048,0.152,19983.79747,12342.85714,13492.04819,1,7
3,4.489,74.3,0.082,0.081,0.119,18167.08861,11551.36778,11600.96386,1,7
4,4.509,74.5,0.084,6.643,6.494,19837.97468,11945.28875,11178.79518,1,8


## 1. Reading a Stream

Instruction: We’re going to read in a stream in the form of .csv files. Create a folder where you will be sending
your .csv files.

In [33]:
#Streaming 파일이 들어갈 폴더 만들기
stream_folder = "stream_folder"
os.makedirs(stream_folder, exist_ok = True)

Instruction: Setup the schema for the stream (you can use the schema from the original data as we did in hw 10)

In [34]:
stream_schema = spark_df.schema

Instruction: Set up the readStream. Be sure to add header = True as you’ll likely be outputting files with a
header and we don’t need to read that in.

In [35]:
stream_df = spark.readStream.option("header", True).schema(stream_schema).csv(stream_folder)

## 2. Transform / Aggregation Step

Instruction: Now, we’ll do two separate things on the stream and join them together:

Instruction: With your stream, use your model transformer to obtain predictions from the incoming data. On
the resulting predictions also create a residual column as noted in the previous section (return
only the label, prediction and residual columns from this part)

In [36]:
#Stream에 학습된 모델 적용
stream_predictions = cv_model.transform(stream_df)

In [38]:
#create residual column
stream_predictions_residual = stream_predictions.withColumn("residual", F.col("label") - F.col("prediction"))

In [39]:
#prediction 결과에서 필요한 칼럼만 선택
stream_prediction_output = stream_predictions_residual.select("label", "prediction", "residual")

Instruction: We can use our stream more than once! With another transformation on the (original) stream, modify the response variable to be called label.

In [40]:
#같은 원본 stream에서 label 컬럼 만들기
stream_label = stream_df.withColumnRenamed("Power_Zone_3", "label")

Instruction: Now join your above transform with this stream based on the label variable which should be common to both!

In [42]:
#두 stream 변환 결과 join
joined_stream = stream_prediction_output.join(stream_label, on = "label")

## 3. Writing Step
Now write your stream to the console using the append output mode.

In [ ]:
query = joined_stream.writeStream.outputMode("append").format("console").start()
query.awaitTermination()

26/04/29 13:40:21 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-f9ce5f7b-6454-412b-93d2-69b11d9fc6f3. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 13:40:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
